In [0]:
from pyspark.sql import functions as F
report_drug = spark.table("report_drug")

# create "report_drug_product_ndc" bridge table to explode product_ndc_arr into individual (report -> product_ndc) relationship rows. Serves as primary linkage layer between FAERS reports and NDC product data supporting downstream joins to drug_products table.
report_drug_product_ndc = (
    report_drug
    .select(
        "report_id",
        "drug_name",
        "drug_characterization",
        "drug_characterization_label",
        F.explode_outer("product_ndc_arr").alias("product_ndc")
    )
    .select(
        "report_id",
        "drug_name",
        "drug_characterization",
        "drug_characterization_label",
        F.upper(F.trim(F.col("product_ndc"))).alias("product_ndc")
    )
    .dropna(subset=["report_id", "product_ndc"])
    .dropDuplicates()
)

report_drug_product_ndc.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("report_drug_product_ndc")
display(report_drug_product_ndc.limit(5))

report_id,drug_name,drug_characterization,drug_characterization_label,product_ndc
25716710,ORENITRAM,1,PRIMARY_SUSPECT_DRUG,66302-300
25716710,BUSPIRONE HCL,2,SECONDARY_SUSPECT_DRUG,72888-066
25716710,LASIX,2,SECONDARY_SUSPECT_DRUG,30698-066
25716710,ALBUTEROL SULFATE,2,SECONDARY_SUSPECT_DRUG,0173-0682
25716710,ALBUTEROL SULFATE,2,SECONDARY_SUSPECT_DRUG,64980-641


In [0]:
# create "report_drug_substance" table to explode substance_name_arr into individual (report -> substance) relationship rows. Serves as primary linkage layer between adverse event reports and active ingredients supporting ingredient-level safety analysis.
report_drug_substance = (
    report_drug
    .select(
        "report_id",
        "drug_name",
        "drug_characterization",
        "drug_characterization_label",
        F.explode_outer("substance_name_arr").alias("substance_name")
    )
    .select(
        "report_id",
        "drug_name",
        "drug_characterization",
        "drug_characterization_label",
        F.upper(F.trim(F.col("substance_name"))).alias("substance_name")
    )
    .dropna(subset=["report_id", "substance_name"])
    .dropDuplicates()
)

report_drug_substance.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("report_drug_substance")
display(report_drug_substance.limit(5))

report_id,drug_name,drug_characterization,drug_characterization_label,substance_name
25979480,CALCIUM,2,SECONDARY_SUSPECT_DRUG,CALCIUM
26041498,IBUPROFEN,2,SECONDARY_SUSPECT_DRUG,IBUPROFEN
26206222,GABAPENTIN,1,PRIMARY_SUSPECT_DRUG,GABAPENTIN
26027703,MAGNESIUM SULFATE,2,SECONDARY_SUSPECT_DRUG,"MAGNESIUM SULFATE, UNSPECIFIED"
26044523,CLOPIDOGREL BISULFATE,1,PRIMARY_SUSPECT_DRUG,CLOPIDOGREL BISULFATE


In [0]:
# create "report_product_links" to link FAERS-derived product_ndc identifiers with the structured NDC drug_products table, establishing many-to-many (report -> product) relationships. *Core integration layer that enables end-to-end graph traversal of adverse event reports to drug products and their active ingredients.*
drug_products_df = spark.table("drug_products")

report_product_links = (
    report_drug_product_ndc.alias("r")
    .join(
        drug_products_df.alias("p"),
        F.col("r.product_ndc") == F.col("p.product_ndc"),
        "inner"
    )
    .select(
        F.col("r.report_id"),
        F.col("r.drug_name"),
        F.col("r.drug_characterization"),
        F.col("r.drug_characterization_label"),
        F.col("p.product_ndc"),
        F.col("p.brand_name"),
        F.col("p.labeler_name"),
        F.col("p.generic_name"),
        F.col("p.dosage_form"),
        F.col("p.product_type")
    )
    .dropDuplicates()
)

report_product_links.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("report_product_links")
display(report_product_links.limit(5))

report_id,drug_name,drug_characterization,drug_characterization_label,product_ndc,brand_name,labeler_name,generic_name,dosage_form,product_type
25716710,ALBUTEROL SULFATE,2,SECONDARY_SUSPECT_DRUG,81894-105,ALBUTEROL SULFATE,"Luoxin Aurovitas Pharma (Chengdu) Co., Ltd.",ALBUTEROL SULFATE,SOLUTION,HUMAN PRESCRIPTION DRUG
25716710,ALBUTEROL SULFATE,2,SECONDARY_SUSPECT_DRUG,53489-176,albuterol sulfate,"Sun Pharmaceutical Industries, Inc.",albuterol sulfate,TABLET,HUMAN PRESCRIPTION DRUG
25716710,ALBUTEROL SULFATE,2,SECONDARY_SUSPECT_DRUG,62135-828,Albuterol Sulfate,"Chartwell RX, LLC",Albuterol Sulfate,SOLUTION,HUMAN PRESCRIPTION DRUG
25716710,SILDENAFIL CITRATE,2,SECONDARY_SUSPECT_DRUG,82347-0210,Sildenafil,Yaral Pharma Inc.,sildenafil citrate,FILM,HUMAN PRESCRIPTION DRUG
25716710,SILDENAFIL CITRATE,2,SECONDARY_SUSPECT_DRUG,59762-0036,Sildenafil Citrate,Mylan Pharmaceuticals Inc.,sildenafil citrate,"TABLET, FILM COATED",HUMAN PRESCRIPTION DRUG


In [0]:
# verify linkage quality between bridge tables
print("Distinct reports in report_drug_product_ndc:",
      report_drug_product_ndc.select("report_id").distinct().count())

print("Distinct reports in report_product_links:",
      report_product_links.select("report_id").distinct().count())

print("Distinct linked products:",
      report_product_links.select("product_ndc").distinct().count())

# output confirms 100% match coverage between adverse event reports and NDC product data

Distinct reports in report_drug_product_ndc: 11884
Distinct reports in report_product_links: 11884
Distinct linked products: 44186
